# A1 Agent Demo
### Make sure all necessary dependencies are installed and environment is running

## Agent setup

In [ ]:
import json
import os
from tasks_executor import autotask_run_by_groups

In [ ]:
ENV_EVAL = False # Eval mode switch. Default is False. Switching to True enables environment feedback loop mode.

In [ ]:
difficulties = ['2'] # Tasks from those difficulties would be used

In [ ]:
tasks_file_name = 'tasks/combined_tasks8_small.json'
prompts_file_name = 'tasks/combined_prompts8_2_small.json' # Path to prompt, containing character initial state

In [ ]:
os.makedirs('results', exist_ok=True)

#### Load previous results

In [ ]:
def load_previous_results(file_name):
    file_name, _ = file_name.split('.')
    file_name = file_name.replace(f'tasks/','/')
    file_name += '_results.json'
    path = 'results' + file_name
    if not os.path.isfile(path):
        return {}, []
    else:
        with open(path, 'r') as f:
            data = json.load(f)
            keys = list(data.keys())
            return data, keys

In [ ]:
previous_data, ignore_ids = load_previous_results(tasks_file_name)

In [ ]:
tasks_file = None
prompts_file = None
with open(tasks_file_name,'r') as f:
    tasks_file = json.load(f)
with open(prompts_file_name,'r') as f:
    prompts_file = json.load(f)

### Agent pipeline

In [ ]:
tasks_run_results = autotask_run_by_groups( # Run A1 agent
    tasks_file, prompts_file,
    experiment_groups = difficulties,
    model='openai/gpt-4.1-mini', 
    ignore_tasks_indexes=ignore_ids,
    env_eval=ENV_EVAL)

In [ ]:
for key, value in tasks_run_results.items(): #Save results
    previous_data[key] = value
results_file_name, _ = tasks_file_name.split('.')
results_file_name = results_file_name.replace(f'tasks/','/')
results_file_name += '_results.json'
path = 'results' + results_file_name
with open(path,'w') as f:
    json.dump(previous_data, f, indent=4)

### Show results

In [ ]:
previous_data, _ = load_previous_results(tasks_file_name) # Load results data

In [ ]:
def show_statistics_per_difficulty(success_per_group,total_per_group, groups):
    group_num = 0
    for group in list(success_per_group.values()):
        group_name = groups[group_num]
        total_monsters = total_per_group[group_name]['monster']
        total_item = total_per_group[group_name]['item']
        total_success_rate = (group['item']+group['monster'])/group['total']
        print(f'''
        Difficulty {group_name}
        Item crafting success rate: {group['item'] / (group['total']-group['monster'])}
        Monster fighting success rate: {group['monster'] / (group['total']-group['item'])}
        Total success rate: {total_success_rate}
        Crafting tasks number: {group['item']} / {total_item}
        Fighting tasks number: {group['monster']} / {total_monsters}
        Total tasks: {group['total']}
        ''')
        group_num += 1

In [ ]:
def show_agent_results(tasks_file, results_file, groups):
    success_per_group = {}
    total_per_group = {}
    for group in groups:
        for task_num, task in enumerate(tasks_file[group]):
            task_index = f'{group}_{task_num}'
            if task_index in results_file:
                
                if group not in success_per_group:
                    success_per_group[group] = {'item': 0, 'monster': 0,'total': 0}
                    total_per_group[group] = {'item': 0, 'monster': 0}
                
                check_agent = results_file[task_index]['agent_check']
                check_env = results_file[task_index]['env_check']
                task_target = results_file[task_index]['task_target']
                task_difficulty = results_file[task_index]['task_difficulty']
                text = f'Task number {task_index} ({task_target}/{task_difficulty}): \n Agent check status {check_agent} \n'
                print(text)
                
                task_type = results_file[task_index]['task_type']
                total_per_group[group][task_type] += 1
                success_per_group[group]['total'] += 1
                if check_env*check_agent == 1:
                    success_per_group[group][task_type] += 1
    
    show_statistics_per_difficulty(success_per_group, total_per_group, groups)

In [ ]:
previous_data_, _ = load_previous_results(tasks_file_name)
show_agent_results(tasks_file,previous_data_,difficulties)